In [2]:
import numpy as np
import pandas as pd
import sqlite3
from scipy import stats

df = pd.read_csv("IBM-HR-Employee-Attrition.csv")

conn = sqlite3.connect(":memory:")
df.to_sql("employee_attrition", conn, index=False, if_exists="replace")

df.head()


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


In [4]:
#PHASE 1: Is overtime associated with performance outcomes, or do heavy workloads mainly increase strain without improving quality of output.


In [6]:
phase1_counts = pd.read_sql_query("""
SELECT
    OverTime,
    PerformanceRating,
    COUNT(*) AS EmployeeCount
FROM employee_attrition
GROUP BY OverTime, PerformanceRating
ORDER BY OverTime, PerformanceRating
""", conn)

velocity_matrix = phase1_counts.pivot(
    index='OverTime',
    columns='PerformanceRating',
    values='EmployeeCount'
).fillna(0)

velocity_matrix = velocity_matrix.div(velocity_matrix.sum(axis=1), axis=0) * 100

print("=== WORKLOAD VELOCITY TEST: OverTime vs. Performance Rating ===")
print(velocity_matrix.round(1).astype(str) + '%')


=== WORKLOAD VELOCITY TEST: OverTime vs. Performance Rating ===
PerformanceRating      3      4
OverTime                       
No                 84.7%  15.3%
Yes                84.4%  15.6%


In [8]:
workload_contingency = pd.crosstab(df['OverTime'], df['PerformanceRating'])

chi2_stat, p_value, dof, expected = stats.chi2_contingency(workload_contingency)

print("=== PHASE 1: WORKLOAD VELOCITY STATISTICAL VALIDATION ===")
print(f"Chi-Square Statistic : {chi2_stat:.4f}")
print(f"Degrees of Freedom   : {dof}")
print(f"P-Value              : {p_value:.6f}")

print("\n=== STRATEGIC VERIFICATION DECISION ===")
if p_value < 0.05:
    print("REJECT NULL HYPOTHESIS: Overtime is statistically associated with performance outcomes.")
    print("Workload intensity appears related to differences in performance-rating distribution.")
else:
    print("FAIL TO REJECT NULL HYPOTHESIS: The performance-rating distribution does not differ significantly by overtime status.")
    print("This dataset does not provide strong evidence that overtime is associated with performance outcomes.")


=== PHASE 1: WORKLOAD VELOCITY STATISTICAL VALIDATION ===
Chi-Square Statistic : 0.0076
Degrees of Freedom   : 1
P-Value              : 0.930471

=== STRATEGIC VERIFICATION DECISION ===
FAIL TO REJECT NULL HYPOTHESIS: The performance-rating distribution does not differ significantly by overtime status.
This dataset does not provide strong evidence that overtime is associated with performance outcomes.


In [10]:
#PHASE 2: Is training frequency statistically associated with performance outcomes.


In [12]:
phase2_counts = pd.read_sql_query("""
SELECT
    TrainingTimesLastYear,
    PerformanceRating,
    COUNT(*) AS EmployeeCount
FROM employee_attrition
GROUP BY TrainingTimesLastYear, PerformanceRating
ORDER BY TrainingTimesLastYear, PerformanceRating
""", conn)

training_matrix = phase2_counts.pivot(
    index='TrainingTimesLastYear',
    columns='PerformanceRating',
    values='EmployeeCount'
).fillna(0)

training_matrix = training_matrix.div(training_matrix.sum(axis=1), axis=0) * 100

print("=== TRAINING VS. EXECUTION: TrainingTimesLastYear vs. Performance Rating ===")
print(training_matrix.round(1).astype(str) + '%')


=== TRAINING VS. EXECUTION: TrainingTimesLastYear vs. Performance Rating ===
PerformanceRating          3      4
TrainingTimesLastYear              
0                      81.5%  18.5%
1                      87.3%  12.7%
2                      83.7%  16.3%
3                      84.9%  15.1%
4                      87.0%  13.0%
5                      84.0%  16.0%
6                      86.2%  13.8%


In [14]:
training_contingency = pd.crosstab(df['TrainingTimesLastYear'], df['PerformanceRating'])

chi2_stat, p_value, dof, expected = stats.chi2_contingency(training_contingency)

print("=== PHASE 2: TRAINING VS. EXECUTION STATISTICAL VALIDATION ===")
print(f"Chi-Square Statistic : {chi2_stat:.4f}")
print(f"Degrees of Freedom   : {dof}")
print(f"P-Value              : {p_value:.6f}")

print("\n=== STRATEGIC VERIFICATION DECISION ===")
if p_value < 0.05:
    print("REJECT NULL HYPOTHESIS: Training frequency is statistically associated with performance outcomes.")
    print("Formal training exposure appears related to differences in performance-rating distribution.")
else:
    print("FAIL TO REJECT NULL HYPOTHESIS: The performance-rating distribution does not differ significantly across training frequencies.")
    print("This dataset does not provide strong evidence that training frequency is associated with performance outcomes.")


=== PHASE 2: TRAINING VS. EXECUTION STATISTICAL VALIDATION ===
Chi-Square Statistic : 1.8580
Degrees of Freedom   : 6
P-Value              : 0.932280

=== STRATEGIC VERIFICATION DECISION ===
FAIL TO REJECT NULL HYPOTHESIS: The performance-rating distribution does not differ significantly across training frequencies.
This dataset does not provide strong evidence that training frequency is associated with performance outcomes.


In [16]:
#PHASE 3: Stability under one manager is often assumed to support performance.
#We are testing whether years with the current manager are statistically associated with performance outcomes.


In [18]:
phase3_counts = pd.read_sql_query("""
SELECT
    YearsWithCurrManager,
    PerformanceRating,
    COUNT(*) AS EmployeeCount
FROM employee_attrition
GROUP BY YearsWithCurrManager, PerformanceRating
ORDER BY YearsWithCurrManager, PerformanceRating
""", conn)

stagnation_matrix = phase3_counts.pivot(
    index='YearsWithCurrManager',
    columns='PerformanceRating',
    values='EmployeeCount'
).fillna(0)

stagnation_matrix = stagnation_matrix.div(stagnation_matrix.sum(axis=1), axis=0) * 100

print("=== LEADERSHIP TENURE ANALYSIS: YearsWithCurrManager vs. Performance Rating ===")
print(stagnation_matrix.round(1).astype(str) + '%')


=== LEADERSHIP TENURE ANALYSIS: YearsWithCurrManager vs. Performance Rating ===
PerformanceRating          3      4
YearsWithCurrManager               
0                      82.9%  17.1%
1                      88.2%  11.8%
2                      88.1%  11.9%
3                      87.3%  12.7%
4                      78.6%  21.4%
5                      83.9%  16.1%
6                      89.7%  10.3%
7                      83.3%  16.7%
8                      78.5%  21.5%
9                      84.4%  15.6%
10                     81.5%  18.5%
11                     81.8%  18.2%
12                     88.9%  11.1%
13                     92.9%   7.1%
14                     80.0%  20.0%
15                     80.0%  20.0%
16                    100.0%   0.0%
17                     85.7%  14.3%


In [20]:
stagnation_contingency = pd.crosstab(df['YearsWithCurrManager'], df['PerformanceRating'])

chi2_stat, p_value, dof, expected = stats.chi2_contingency(stagnation_contingency)

print("=== PHASE 3: LEADERSHIP TENURE STATISTICAL VALIDATION ===")
print(f"Chi-Square Statistic : {chi2_stat:.4f}")
print(f"Degrees of Freedom   : {dof}")
print(f"P-Value              : {p_value:.6f}")

print("\n=== STRATEGIC VERIFICATION DECISION ===")
if p_value < 0.05:
    print("REJECT NULL HYPOTHESIS: Years with the current manager are statistically associated with performance outcomes.")
    print("Manager-tenure differences appear related to performance-rating distribution.")
else:
    print("FAIL TO REJECT NULL HYPOTHESIS: The performance-rating distribution does not differ significantly by years with current manager.")
    print("This dataset does not provide strong evidence that manager tenure is associated with performance outcomes.")


=== PHASE 3: LEADERSHIP TENURE STATISTICAL VALIDATION ===
Chi-Square Statistic : 13.8433
Degrees of Freedom   : 17
P-Value              : 0.678165

=== STRATEGIC VERIFICATION DECISION ===
FAIL TO REJECT NULL HYPOTHESIS: The performance-rating distribution does not differ significantly by years with current manager.
This dataset does not provide strong evidence that manager tenure is associated with performance outcomes.


In [22]:
#PHASE 4: Time spent in the same role may influence whether employees continue performing at a high level.
#We are testing whether years in current role are statistically associated with performance outcomes.


In [24]:
phase4_counts = pd.read_sql_query("""
SELECT
    YearsInCurrentRole,
    PerformanceRating,
    COUNT(*) AS EmployeeCount
FROM employee_attrition
GROUP BY YearsInCurrentRole, PerformanceRating
ORDER BY YearsInCurrentRole, PerformanceRating
""", conn)

mobility_matrix = phase4_counts.pivot(
    index='YearsInCurrentRole',
    columns='PerformanceRating',
    values='EmployeeCount'
).fillna(0)

mobility_matrix = mobility_matrix.div(mobility_matrix.sum(axis=1), axis=0) * 100

print("=== ROLE TENURE ANALYSIS: YearsInCurrentRole vs. Performance Rating ===")
print(mobility_matrix.round(1).astype(str) + '%')


=== ROLE TENURE ANALYSIS: YearsInCurrentRole vs. Performance Rating ===
PerformanceRating        3      4
YearsInCurrentRole               
0                    84.0%  16.0%
1                    87.7%  12.3%
2                    87.6%  12.4%
3                    84.4%  15.6%
4                    81.7%  18.3%
5                    72.2%  27.8%
6                    86.5%  13.5%
7                    85.6%  14.4%
8                    83.1%  16.9%
9                    83.6%  16.4%
10                   86.2%  13.8%
11                   81.8%  18.2%
12                   70.0%  30.0%
13                   71.4%  28.6%
14                   81.8%  18.2%
15                   75.0%  25.0%
16                   71.4%  28.6%
17                  100.0%   0.0%
18                  100.0%   0.0%


In [26]:
mobility_contingency = pd.crosstab(df['YearsInCurrentRole'], df['PerformanceRating'])

chi2_stat, p_value, dof, expected = stats.chi2_contingency(mobility_contingency)

print("=== PHASE 4: ROLE TENURE STATISTICAL VALIDATION ===")
print(f"Chi-Square Statistic : {chi2_stat:.4f}")
print(f"Degrees of Freedom   : {dof}")
print(f"P-Value              : {p_value:.6f}")

print("\n=== STRATEGIC VERIFICATION DECISION ===")
if p_value < 0.05:
    print("REJECT NULL HYPOTHESIS: Years in current role are statistically associated with performance outcomes.")
    print("Role tenure appears related to differences in performance-rating distribution.")
else:
    print("FAIL TO REJECT NULL HYPOTHESIS: The performance-rating distribution does not differ significantly by years in current role.")
    print("This dataset does not provide strong evidence that role tenure is associated with performance outcomes.")


=== PHASE 4: ROLE TENURE STATISTICAL VALIDATION ===
Chi-Square Statistic : 14.8397
Degrees of Freedom   : 18
P-Value              : 0.672946

=== STRATEGIC VERIFICATION DECISION ===
FAIL TO REJECT NULL HYPOTHESIS: The performance-rating distribution does not differ significantly by years in current role.
This dataset does not provide strong evidence that role tenure is associated with performance outcomes.


In [28]:
#PHASE : Is percent salary hike statistically associated with performance rating.


In [30]:
phase5_counts = pd.read_sql_query("""
SELECT
    PercentSalaryHike,
    PerformanceRating,
    COUNT(*) AS EmployeeCount
FROM employee_attrition
GROUP BY PercentSalaryHike, PerformanceRating
ORDER BY PercentSalaryHike, PerformanceRating
""", conn)

merit_matrix = phase5_counts.pivot(
    index='PercentSalaryHike',
    columns='PerformanceRating',
    values='EmployeeCount'
).fillna(0)

merit_matrix = merit_matrix.div(merit_matrix.sum(axis=1), axis=0) * 100

print("=== MERITOCRACY ALIGNMENT: PercentSalaryHike vs. Performance Rating ===")
print(merit_matrix.round(1).astype(str) + '%')


=== MERITOCRACY ALIGNMENT: PercentSalaryHike vs. Performance Rating ===
PerformanceRating       3       4
PercentSalaryHike                
11                 100.0%    0.0%
12                 100.0%    0.0%
13                 100.0%    0.0%
14                 100.0%    0.0%
15                 100.0%    0.0%
16                 100.0%    0.0%
17                 100.0%    0.0%
18                 100.0%    0.0%
19                 100.0%    0.0%
20                   0.0%  100.0%
21                   0.0%  100.0%
22                   0.0%  100.0%
23                   0.0%  100.0%
24                   0.0%  100.0%
25                   0.0%  100.0%


In [32]:
merit_contingency = pd.crosstab(df['PercentSalaryHike'], df['PerformanceRating'])

chi2_stat, p_value, dof, expected = stats.chi2_contingency(merit_contingency)

print("=== PHASE 5: MERITOCRACY ALIGNMENT STATISTICAL VALIDATION ===")
print(f"Chi-Square Statistic : {chi2_stat:.4f}")
print(f"Degrees of Freedom   : {dof}")
print(f"P-Value              : {p_value:.6f}")

print("\n=== STRATEGIC VERIFICATION DECISION ===")
if p_value < 0.05:
    print("REJECT NULL HYPOTHESIS: Percent salary hike is statistically associated with performance rating.")
    print("Compensation adjustments appear related to performance-rating outcomes.")
else:
    print("FAIL TO REJECT NULL HYPOTHESIS: The salary-hike distribution does not differ significantly by performance rating.")
    print("This dataset does not provide strong evidence that percent salary hike is associated with performance rating.")


=== PHASE 5: MERITOCRACY ALIGNMENT STATISTICAL VALIDATION ===
Chi-Square Statistic : 1470.0000
Degrees of Freedom   : 14
P-Value              : 0.000000

=== STRATEGIC VERIFICATION DECISION ===
REJECT NULL HYPOTHESIS: Percent salary hike is statistically associated with performance rating.
Compensation adjustments appear related to performance-rating outcomes.
